In [1]:
import json
import pandas as pd

with open("xscores_all_data.json") as f:
    all_data = json.load(f)
with open("xscores_data.json") as f:
    lookups = json.load(f)["Lookups"]

# Index -> label maps for the coded columns
maps = {
    name: dict(zip(lk[0]["Index"], lk[0]["Label"]))
    for name, lk in lookups.items()
    if name != "Player"
}
maps["Opposition"] = maps["Team"]

In [2]:
# One row per shot: each player's column-oriented "Data" becomes a frame, then concatenated
frames = []
for file_data in all_data.values():
    for player_id, p in file_data.items():
        df = pd.DataFrame(p["Data"])
        df.insert(0, "player_id", player_id)
        df.insert(1, "player", p["Player"]["Label"][0])
        frames.append(df)

shots = pd.concat(frames, ignore_index=True)

# Decode the coded columns to labels
for col in ["Season", "Team", "Opposition", "Distance", "ShotType", "Pocket"]:
    shots[col] = shots[col].map(maps[col])

print(shots.shape)
shots.head()

(64550, 13)


,player_id,player,Season,Round,Team,Opposition,Distance,Pocket,ShotType,x,y,Score,xScore
0,1407800,Jaidyn Stephenson,2021,1,North Melbourne,Port Adelaide,30 to <40 m,Pocket,Set Position,29.0,-26,1,3.4
1,1407800,Jaidyn Stephenson,2021,3,North Melbourne,Western Bulldogs,30 to <40 m,Corridor,Set Position,34.0,-11,6,4.6
2,1407800,Jaidyn Stephenson,2021,3,North Melbourne,Western Bulldogs,15 to <30 m,Pocket,Set Position,13.0,-21,6,3.6
3,1407800,Jaidyn Stephenson,2021,5,North Melbourne,Geelong,40 to <50 m,Pocket,Set Position,25.0,37,0,2.1
4,1407800,Jaidyn Stephenson,2021,5,North Melbourne,Geelong,40 to <50 m,Corridor,Set Position,45.0,19,1,3.0


In [3]:
shots.to_csv("afl_shots.csv")